# Train Coco's voice (Piper, high quality)

Fine-tunes Piper's **en_US lessac high** voice on Coco's dataset. Free Colab GPU (T4) is enough.

**Before you start:** upload `coco-voice-dataset.zip` to your Google Drive in a folder called
`coco-voice` (so: `My Drive/coco-voice/coco-voice-dataset.zip`).

**How to run:** Runtime > Change runtime type > **T4 GPU**. Then run the cells top to bottom.
Training runs in the background and keeps checkpoints on Colab's disk; the newest one is copied to
Drive every 45 minutes (cell 4c), and the Stop cell saves before stopping. If Colab disconnects (it
will, on the free tier), run everything again: training resumes from the last copy on Drive.

**When is it done?** Listen with the "Listen" cell every hour or so. Stop when it sounds right
(usually a few hundred epochs past the start). More is not always better: pick by ear.

In [ ]:
# ---- Settings ----
EXTRA_EPOCHS = 300        # Epochs to train beyond the starting checkpoint (raise and re-run to train longer)
BATCH_SIZE = 16           # Lower to 8 if you get "CUDA out of memory"
PRECISION = "16-mixed"    # About 2x faster on a T4. If losses show "nan", change to "32" and re-run (it resumes)
MODE = "resume"           # "resume": continue the lessac checkpoint (recommended by Piper's docs)
                          # "warmstart": copy only its weights and start a fresh schedule (use if "resume" errors)
PIPER_COMMIT = "39ac0a04ff01120b1cb7867d21723fa320305b03"  # piper1-gpl version this notebook was written for
DRIVE_DIR = "/content/drive/MyDrive/coco-voice"
VOICE_NAME = "coco"

# Storage: checkpoints (~1 GB each, 3 kept: see 4d) and the log live on Colab's local disk. Every
# SYNC_MINUTES the newest one is copied to Drive and the previous copy is deleted permanently (not to
# the Trash, which counts against Drive storage). A disconnect loses at most SYNC_MINUTES of training.
LOCAL_RUNS = "/content/runs"
DRIVE_CKPT_DIR = f"{DRIVE_DIR}/checkpoint"
LOG = "/content/train.log"
SYNC_MINUTES = 45

# Newest checkpoint that is completely written (a file being saved, or cut off by a disconnect, is a
# truncated ZIP). Looks locally, then at the Drive copy, then at the older Drive layout (runs/).
import glob, os, zipfile
def newest_checkpoint():
    paths = (glob.glob(f"{LOCAL_RUNS}/**/*.ckpt", recursive=True) + glob.glob(f"{DRIVE_CKPT_DIR}/*.ckpt")
             + glob.glob(f"{DRIVE_DIR}/runs/**/*.ckpt", recursive=True))
    for path in sorted(paths, key=os.path.getmtime, reverse=True):
        if zipfile.is_zipfile(path):
            return path
    return None

# Lets this notebook's Python import Piper. It started before the install, so it doesn't see the editable
# package. Python also remembers a search-path folder that was missing when it last looked, and never
# checks it again, so the remembered "missing" is cleared each time.
import importlib, sys
def use_piper():
    src = "/content/piper1-gpl/src"
    if not os.path.isdir(f"{src}/piper"):
        raise RuntimeError("Piper isn't installed on this runtime (a new or reset runtime starts empty). "
                           "Run cell 2 (Install) first, then this cell again.")
    sys.path_importer_cache.pop(src, None)
    if src not in sys.path:
        sys.path.insert(0, src)
    importlib.invalidate_caches()

In [ ]:
# ---- 1. GPU and Google Drive ----
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount("/content/drive")
import os
assert os.path.exists(f"{DRIVE_DIR}/coco-voice-dataset.zip"), f"Upload coco-voice-dataset.zip to {DRIVE_DIR} first"


In [ ]:
# ---- 2. Install Piper training (pinned version) ----
!apt-get -qq update && apt-get -qq install -y espeak-ng build-essential cmake ninja-build > /dev/null
%cd /content
!test -d piper1-gpl || git clone -q https://github.com/OHF-Voice/piper1-gpl.git
%cd /content/piper1-gpl
!git checkout -q {PIPER_COMMIT}
!pip -q install -e '.[train]'
!bash ./build_monotonic_align.sh
# The in-place build of the eSpeak bridge runs setup.py directly, so its build tools must be in this Python
# (pip install -e only had them in a temporary build environment)
!pip -q install scikit-build setuptools wheel cmake ninja
!python3 setup.py build_ext --inplace > /tmp/build_ext.log 2>&1 || (tail -30 /tmp/build_ext.log; false)
!python3 -c "from piper import espeakbridge; from piper.train.vits import monotonic_align; print('eSpeak bridge and monotonic_align OK')"

In [ ]:
# ---- 3. Dataset (copied to the fast local disk) and the lessac high checkpoint (cached on Drive) ----
!test -d /content/data/wavs || (mkdir -p /content/data && unzip -q "{DRIVE_DIR}/coco-voice-dataset.zip" -d /content/data)
!echo "clips: $(wc -l < /content/data/metadata.csv)"
LESSAC = f"{DRIVE_DIR}/lessac-high.ckpt"
!test -f "{LESSAC}" || wget -q -O "{LESSAC}" "https://huggingface.co/datasets/rhasspy/piper-checkpoints/resolve/main/en/en_US/lessac/high/epoch%3D2218-step%3D838782.ckpt?download=true"
!ls -la "{LESSAC}"

In [ ]:
# ---- 4a. Piper "high" model shape, and a check that the lessac checkpoint fits it ----
use_piper()
# (The old trainer's --quality high. The new trainer's defaults are medium. "1" is quoted on purpose:
#  the model compares resblock == "1", and an unquoted 1 would quietly build the wrong vocoder.)
HIGH_YAML = """
model:
  resblock: "1"
  resblock_kernel_sizes: [3, 7, 11]
  resblock_dilation_sizes: [[1, 3, 5], [1, 3, 5], [1, 3, 5]]
  upsample_rates: [8, 8, 2, 2]
  upsample_initial_channel: 512
  upsample_kernel_sizes: [16, 16, 4, 4]
"""
open("/content/high.yaml", "w").write(HIGH_YAML)
import torch, yaml
from piper.train.vits.lightning import VitsModel
model = VitsModel(**yaml.safe_load(HIGH_YAML)["model"], mos_metric=None)
own = model.state_dict()
# Official checkpoint from rhasspy (Piper's author); old Lightning format, so weights_only=False
ckpt = torch.load(LESSAC, map_location="cpu", weights_only=False)["state_dict"]
fits = [k for k, v in ckpt.items() if k in own and own[k].shape == v.shape]
print(f"{len(fits)} of {len(ckpt)} checkpoint weights fit the high model")
assert len(fits) == len(ckpt), "Shape mismatch: don't train, the model settings are wrong"
del model, ckpt

In [ ]:
# ---- 4b. One-time: a cleaned copy of the lessac checkpoint that the new trainer accepts ----
use_piper()
# 1. Lightning resumes with torch.load(weights_only=True), which refuses non-data objects: the 2023
#    checkpoint stores a pathlib.PosixPath, so paths become plain strings.
# 2. On resume, Lightning re-applies the checkpoint's saved settings as --model.* options. The 2023
#    checkpoint has settings the new trainer dropped (sample_bytes, ...), so only the ones it accepts are
#    kept. batch_size and num_symbols are filled in from the data settings and can't be given directly.
import os, inspect, pathlib, torch
from piper.train.vits.lightning import VitsModel
LESSAC_CLEAN = f"{DRIVE_DIR}/lessac-high-clean-v2.ckpt"
if not os.path.exists(LESSAC_CLEAN):
    ck = torch.load(LESSAC, map_location="cpu", weights_only=False)  # official rhasspy checkpoint
    def plain(o):
        if isinstance(o, pathlib.PurePath):
            return str(o)
        if isinstance(o, dict):
            return {k: plain(v) for k, v in o.items()}
        if isinstance(o, (list, tuple)):
            return type(o)(plain(v) for v in o)
        return o
    ck = {k: v if k == "state_dict" else plain(v) for k, v in ck.items()}
    accepted = set(inspect.signature(VitsModel.__init__).parameters) - {"self", "kwargs", "dataset", "batch_size", "num_symbols"}
    old = ck.get("hyper_parameters", {})
    ck["hyper_parameters"] = {k: v for k, v in old.items() if k in accepted}
    print("dropped old settings:", sorted(set(old) - set(ck["hyper_parameters"])))
    torch.save(ck, LESSAC_CLEAN)
    del ck
torch.load(LESSAC_CLEAN, map_location="cpu", weights_only=True)  # The same safe load Lightning does
print("Cleaned checkpoint passes the safe loader:", LESSAC_CLEAN)


In [ ]:
# ---- 4c. Drive sync (run once per session, before training; asks for Google access once) ----
import os, glob, shutil, threading, time, zipfile, torch
from google.colab import auth
auth.authenticate_user()  # Needed to delete old copies permanently (via the Drive API, skipping the Trash)
from googleapiclient.discovery import build
drive_api = build("drive", "v3", cache_discovery=False)
os.makedirs(DRIVE_CKPT_DIR, exist_ok=True)

def sync_now():
    local = [p for p in sorted(glob.glob(f"{LOCAL_RUNS}/**/*.ckpt", recursive=True), key=os.path.getmtime, reverse=True)
             if zipfile.is_zipfile(p)]
    if not local:
        return "nothing to sync yet"
    epoch = torch.load(local[0], map_location="cpu", weights_only=True, mmap=True)["epoch"]
    dst = f"{DRIVE_CKPT_DIR}/coco-epoch{epoch}.ckpt"
    if not os.path.exists(dst):
        shutil.copy2(local[0], dst + ".part")   # copy2 keeps the time it was saved, so "newest" stays right
        os.rename(dst + ".part", dst)
        if not zipfile.is_zipfile(dst):
            return f"copy of epoch {epoch} looks incomplete; keeping the older copies"
    # Delete the older copies permanently: only "coco-epoch*.ckpt" files in this same Drive folder
    me = drive_api.files().list(q=f"name = '{os.path.basename(dst)}' and trashed = false",
                                fields="files(id,parents)").execute()["files"]
    deleted = []
    if me:
        folder = me[0]["parents"][0]
        for f in drive_api.files().list(q=f"'{folder}' in parents and name contains 'coco-epoch' and trashed = false",
                                        fields="files(id,name)").execute()["files"]:
            if f["name"] != os.path.basename(dst) and f["name"].endswith((".ckpt", ".ckpt.part")):
                drive_api.files().delete(fileId=f["id"]).execute()
                deleted.append(f["name"])
    return f"epoch {epoch} saved to Drive" + (f", removed {', '.join(deleted)}" if deleted else "")

def _sync_loop():
    while True:
        time.sleep(SYNC_MINUTES * 60)
        try:
            msg = sync_now()
        except Exception as e:
            msg = f"sync failed: {e}"
        with open("/content/sync.log", "a") as f:
            f.write(f"{time.strftime('%H:%M')} {msg}\n")

if not any(t.name == "coco-sync" for t in threading.enumerate()):
    threading.Thread(target=_sync_loop, name="coco-sync", daemon=True).start()
print(f"Drive sync running: every {SYNC_MINUTES} min to {DRIVE_CKPT_DIR}")

In [ ]:
# ---- 4d. Train IN THE BACKGROUND (resumes automatically) ----
# A background process, so the notebook stays free for Progress, Listen and Stop (Colab runs one cell
# at a time). Checkpoints go to Colab's local disk; the sync cell copies the newest one to Drive.
import os, subprocess
LESSAC_EPOCH = 2218
# Keep 3 checkpoints on disk, not 11. Piper's trainer always adds its own savers (Lightning appends
# them even if a config lists others): the 5 best by audio error (val_mel), the 5 best by predicted
# listening quality (val_mos), and last.ckpt. Cut both "5 best" to 1: best val_mel, best val_mos, latest.
MAIN = "/content/piper1-gpl/src/piper/train/__main__.py"
src = open(MAIN).read()
if src.count("save_top_k=5") == 2:
    open(MAIN, "w").write(src.replace("save_top_k=5", "save_top_k=1"))
elif "save_top_k=5" in src or src.count("save_top_k=1") < 3:
    raise RuntimeError("Piper's checkpoint settings aren't where this notebook expects; nothing changed.")
if subprocess.run(["pgrep", "-f", "piper.train fit"], capture_output=True).returncode == 0:
    print("Training is already running. Use the Progress cell to watch it.")
else:
    resume_from = newest_checkpoint()
    if resume_from:
        start = f"--ckpt_path '{resume_from}'"
        print("Resuming our own run from", resume_from)
    elif MODE == "resume":
        start = f"--ckpt_path '{LESSAC_CLEAN}'"
        print("Starting from the lessac high checkpoint (resume mode)")
    else:
        start = f"--model.warmstart_ckpt '{LESSAC_CLEAN}' --model.learning_rate 0.0001"
        print("Starting from the lessac high weights (warmstart mode)")
    max_epochs = (LESSAC_EPOCH if MODE == "resume" else 0) + EXTRA_EPOCHS
    cmd = (f"python3 -m piper.train fit --config /content/high.yaml "
           f"--data.voice_name {VOICE_NAME} --data.csv_path /content/data/metadata.csv "
           f"--data.audio_dir /content/data/wavs --model.sample_rate 22050 --data.espeak_voice en-us "
           f"--data.cache_dir /content/cache --data.config_path '{DRIVE_DIR}/config.json' "
           f"--data.batch_size {BATCH_SIZE} --data.length_bucketing true "
           f"--trainer.accelerator gpu --trainer.devices 1 --trainer.precision {PRECISION} "
           f"--trainer.max_epochs {max_epochs} --trainer.default_root_dir '{LOCAL_RUNS}' {start}")
    with open(LOG, "a") as log:
        log.write(f"\n===== starting: {cmd}\n")
    subprocess.Popen(f"{cmd} >> '{LOG}' 2>&1", shell=True, cwd="/content/piper1-gpl", start_new_session=True)
    print("Training started in the background. Log:", LOG)

In [ ]:
# ---- 4e. Progress (run any time; re-run to refresh) ----
import subprocess, os, time, torch
running = subprocess.run(["pgrep", "-f", "piper.train fit"], capture_output=True).returncode == 0
print("Training is", "RUNNING" if running else "NOT running")
ck = newest_checkpoint()
if ck:
    epoch = torch.load(ck, map_location="cpu", weights_only=True, mmap=True)["epoch"]
    print(f"newest complete checkpoint: epoch {epoch} ({epoch - 2218} past the lessac start), "
          f"saved {time.time() - os.path.getmtime(ck):.0f}s ago: {ck}")
if os.path.exists("/content/sync.log"):
    print("Drive sync:", open("/content/sync.log").read().strip().splitlines()[-1])
!tail -c 4000 /content/train.log 2>/dev/null | tr '\r' '\n' | grep -v "^\s*$" | tail -8

In [ ]:
# ---- 4f. Stop training and save the newest checkpoint to Drive (run 4d again to resume) ----
import time
!pkill -f "piper.train fit" && echo "Stopped training." || echo "Training wasn't running."
time.sleep(5)
print(sync_now())

## Listen

Run this any time, also while training runs in the background. It exports the latest saved
checkpoint and speaks a few lines Coco would actually say.

In [ ]:
# ---- 5. Listen to the latest checkpoint ----
use_piper()
import glob, os, wave
from IPython.display import Audio, display
latest = newest_checkpoint()
assert latest, "No complete checkpoint yet: wait for the first epoch to finish"
print("Checkpoint:", latest)
!python3 -m piper.train.export_onnx --checkpoint "{latest}" --output-file /content/listen.onnx
!cp "{DRIVE_DIR}/config.json" /content/listen.onnx.json
from piper import PiperVoice
voice = PiperVoice.load("/content/listen.onnx", config_path="/content/listen.onnx.json")
for i, line in enumerate([
    "Hey. What are we fixing today?",
    "Your shopping list has eggs, oat milk, flour and coffee beans. Butter wasn't on it, so I added it.",
    "Black jeans and a white tee? Clean. The boots are doing the heavy lifting, babe.",
    "Okay. Breathe. Your notes are all still here. Nothing's gone.",
]):
    with wave.open(f"/content/listen_{i}.wav", "wb") as wf:
        voice.synthesize_wav(line, wf)
    print(line); display(Audio(f"/content/listen_{i}.wav"))

## Export the finished voice

When you like how it sounds, run this. Then download the two files from `My Drive/coco-voice/export/`
and put them in the repo's `models/piper/` folder.

In [ ]:
# ---- 6. Export ----
import glob, os, shutil
latest = newest_checkpoint()
assert latest, "No complete checkpoint yet: wait for the first epoch to finish"
os.makedirs(f"{DRIVE_DIR}/export", exist_ok=True)
out = f"{DRIVE_DIR}/export/en_US-{VOICE_NAME}-high.onnx"
!python3 -m piper.train.export_onnx --checkpoint "{latest}" --output-file "{out}"
shutil.copy(f"{DRIVE_DIR}/config.json", out + ".json")
print("Exported from", latest); !ls -la "{DRIVE_DIR}/export"